In [ ]:
import os
import yaml
from tqdm import tqdm
import numpy as np
import polars as pl

In [ ]:
DATA_DIR = 'PATH_TO_FILE'

# Create input for VEP and annotation pipeline

In [ ]:
# PROTEINGYM_URL = "https://marks.hms.harvard.edu/proteingym/ProteinGym_v1.3/DMS_substitutions.csv"
# PROTEINGYM_CA_BUNDLE = "certs/incommon_rsa_ov_ssl_ca3_chain.pem"  # completes InCommon chain the server omits

# !wget --ca-certificate={PROTEINGYM_CA_BUNDLE} -P {DATA_DIR}/ {PROTEINGYM_URL}

In [ ]:
# PROTEINGYM_URL = "https://marks.hms.harvard.edu/proteingym/ProteinGym_v1.3/DMS_ProteinGym_substitutions.zip"
# PROTEINGYM_CA_BUNDLE = "certs/incommon_rsa_ov_ssl_ca3_chain.pem"  # completes InCommon chain the server omits

# !wget --ca-certificate={PROTEINGYM_CA_BUNDLE} -P {DATA_DIR}/ {PROTEINGYM_URL}

# !unzip -o {DATA_DIR}/DMS_ProteinGym_substitutions.zip -d {DATA_DIR}/

In [ ]:
dms_dir = f'{DATA_DIR}/DMS_ProteinGym_substitutions/'
# dms_dir = f'{DATA_DIR}/zero_shot_substitutions_scores/'

pg_list = []
for filename in tqdm(os.listdir(dms_dir)):
    if 'HUMAN' in filename:
        if filename.endswith('.csv'):
            temp = pl.read_csv(f'{dms_dir}{filename}')
            big_name = filename.split('.')[0]
            temp = temp.with_columns(
                pl.lit(big_name.split('_')[0]).alias('protein_name'),
                # pl.lit('_'.join(big_name.split('_')[2:])).alias('experiment_name'),
                pl.lit(filename.split('.')[0]).alias('file_name'),
                )
            pg_list.append(temp)

pgdf = pl.concat(pg_list)

# ProteinGym filenames encode a UniProt entry-name token (e.g. "P53", "A4"), not an
# HGNC gene symbol -- ~45% of human assays disagree (P53 -> TP53, A4 -> APP, ...).
# Resolve to the authoritative HGNC symbol via proteingym_uniprot_to_hgnc.csv (built
# from the UniProt REST API), keyed on this same filename token.
gene_map = pl.read_csv('proteingym_uniprot_to_hgnc.csv').select(['filename_token', 'hgnc_symbol'])
pgdf = (
    pgdf
    .join(gene_map, left_on='protein_name', right_on='filename_token', how='left')
    .rename({'hgnc_symbol': 'gene_name'})
)

print(f"Number of unique proteins in ProteinGym: {pgdf['protein_name'].n_unique()}")
pgdf

In [ ]:
ginfo = (
    pl.read_parquet('PATH_TO_FILE')
    .filter(pl.col('ensembl_canonical') == True)
    .filter(pl.col('gene_type') == 'protein_coding')
    .rename({
        'gene_stable_id': 'region',
        'gene_name': 'protein_name',  # HGNC-symbol-sourced column, broader coverage than uniprotkb_gene_name_symbol
    })
    .select(['region', 'protein_name', 'gene_start_(bp)'])
    .drop_nulls(subset=['protein_name'])
    .unique()

    .join(
        pgdf.select('gene_name').unique().rename({'gene_name': 'protein_name'}),
        on='protein_name',
        how='semi',
    )

    # Some symbols resolve to >1 region: one on the primary assembly plus copies on GRCh38 alt-haplotype/patch scaffolds (e.g. HLA-A's 7 alt-haplotype copies, or PTEN/SRC/HRAS's single alt copy). Alt-scaffold coordinates are local to a short contig, so the copy with the largest gene_start is reliably the primary-assembly one.
    .sort('gene_start_(bp)', descending=True)
    .group_by('protein_name', maintain_order=True)
    .agg(pl.col('region').first())
)
ginfo

In [ ]:
pgdf = (
    pgdf
    .select(['gene_name', 'mutant', 'file_name', 'protein_name', 'DMS_score', 'DMS_score_bin'])
    .join(
        ginfo,
        left_on='gene_name',
        right_on='protein_name',
        how='left',
        validate='m:1'
    )
)

pgdf

## Annotate with popEVE to get `chrom:pos:ref:alt`

In [ ]:
pe = (
    pl.read_csv(
        'PATH_TO_FILE', 
        separator='\t',
        ignore_errors=True,
        schema_overrides={'#CHROM': pl.Utf8},
    )
    .with_columns(
        # Extract everything after 'protein=' until the next ';' or end of string
        id = pl.concat_str([pl.col('#CHROM'), pl.col('POS').cast(pl.Utf8), pl.col('REF'), pl.col('ALT')], separator=':'),
        protein = pl.col('INFO').str.extract(r"protein=([^;]+)"),
        
        # Extract everything after 'gene=' until the next ';' or end of string
        gene_name = pl.col('INFO').str.extract(r"gene=([^;]+)"),
        mutant = pl.col('INFO').str.extract(r"mutant=([^;]+)"),
        
        popeve_score = pl.col('INFO').str.extract(r"popEVE=([^;]+)").cast(pl.Float32),
        esm1v = pl.col('INFO').str.extract(r"ESM1v=([^;]+)").cast(pl.Float32)
    )
    .with_columns(
        ref_aa=pl.col('mutant').str.slice(0, 1),
        alt_aa=pl.col('mutant').str.slice(-1, 1),
        prot_pos=pl.col('mutant').str.slice(1, pl.col('mutant').str.len_chars() - 2),
    )
    .with_columns(
        amino_acids=pl.col('ref_aa') + '/' + pl.col('alt_aa'),
    )
    .select(['gene_name', 'id', 'mutant', 'amino_acids', 'prot_pos', 'ref_aa', 'alt_aa', 'popeve_score', 'esm1v'])
    .drop_nulls(subset=['gene_name', 'mutant', 'id'])
    .unique()

    .join(
        ginfo.select(['region', 'protein_name']).unique(),
        left_on='gene_name',
        right_on='protein_name',
        how='semi',
    )
)

pe

In [ ]:
# popEVE sometimes scores a different RefSeq isoform than the one the DMS assay used
# (e.g. PTEN-Long's 173-residue N-terminal extension vs canonical PTEN), which shifts
# every position by a constant amount and silently breaks the (gene_name, mutant) join
# even though the amino-acid change is otherwise identical. Find, per gene, the constant
# offset that best reconciles pgdf's wildtype residues with popEVE's at the shifted
# position, and only apply it where the fix is unambiguous (>=90% of positions agree).
SEARCH_RANGE = 250
MIN_MATCH_FRAC = 0.9

pg_pos_wt = (
    pgdf.filter(~pl.col('mutant').str.contains(':'))
    .select('gene_name', 'mutant').unique()
    .with_columns(
        wt=pl.col('mutant').str.slice(0, 1),
        pos=pl.col('mutant').str.extract(r'(\d+)', 1).cast(pl.Int64),
    )
)
pe_pos_wt = (
    pe.select('gene_name', 'mutant').unique()
    .with_columns(
        wt=pl.col('mutant').str.slice(0, 1),
        pos=pl.col('mutant').str.extract(r'(\d+)', 1).cast(pl.Int64),
    )
)

offset_rows = []
for gene in pg_pos_wt['gene_name'].unique().drop_nulls().to_list():
    pg_g = pg_pos_wt.filter(pl.col('gene_name') == gene).select('pos', 'wt').unique()
    pe_g = pe_pos_wt.filter(pl.col('gene_name') == gene).select('pos', 'wt').unique().rename({'wt': 'wt_pe'})
    if pg_g.height == 0 or pe_g.height == 0:
        continue
    best_offset, best_frac = 0, -1.0
    for offset in range(-SEARCH_RANGE, SEARCH_RANGE + 1):
        matched = (
            pg_g.with_columns((pl.col('pos') + offset).alias('pos'))
            .join(pe_g, on='pos', how='inner')
            .filter(pl.col('wt') == pl.col('wt_pe'))
            .height
        )
        frac = matched / pg_g.height
        if frac > best_frac:
            best_offset, best_frac = offset, frac
    offset_rows.append((gene, best_offset, best_frac))

gene_offsets = (
    pl.DataFrame(offset_rows, schema=['gene_name', 'offset', 'match_frac'], orient='row')
    .with_columns(
        pl.when(pl.col('match_frac') >= MIN_MATCH_FRAC).then(pl.col('offset')).otherwise(0).alias('offset')
    )
)
gene_offsets.filter(pl.col('offset') != 0).sort('offset')

In [ ]:
pgdf = (
    pgdf
    .join(gene_offsets.select('gene_name', 'offset'), on='gene_name', how='left')
    .with_columns(pl.col('offset').fill_null(0))
    .with_columns(
        mutant_shifted=pl.when(pl.col('mutant').str.contains(':') | (pl.col('offset') == 0))
        .then(pl.col('mutant'))
        .otherwise(
            pl.col('mutant').str.slice(0, 1)
            + (pl.col('mutant').str.extract(r'(\d+)', 1).cast(pl.Int64) + pl.col('offset')).cast(pl.Utf8)
            + pl.col('mutant').str.extract(r'\d+(.+)$', 1)
        )
    )
)
pgdf

In [ ]:
pe['gene_name'].n_unique()

In [ ]:
pgdf_anno = (
    pgdf
    .join(
        pe,
        left_on=['gene_name', 'mutant_shifted'],
        right_on=['gene_name', 'mutant'],
        how='inner',
        # validate='1:1'
    )
    .with_columns(
        pl.col('id').str.split_exact(':', 3).struct.rename_fields(['chrom', 'pos', 'ref', 'alt']).alias('id_parts')
    )
    .unnest('id_parts')
    .with_columns(pl.col('pos').cast(pl.Int64))

    # .drop_nulls(subset=['id'])
)

pgdf_anno

In [ ]:
pgdf_anno.write_parquet(f'{DATA_DIR}/SNVs_popeve_annotated.parquet', compression='zstd')

## Annotate with all other missense annotations

In [ ]:
import sys
sys.path.insert(0, '../annotations')  # add_more_annotations.py lives there
import add_more_annotations as ann

In [ ]:
ann.main(
    f"{DATA_DIR}/SNVs_popeve_annotated.parquet",
    fill_null_defaults_path = "../annotations/fill_null_defaults.yaml",
    download_dir = f"{DATA_DIR}/tmp",
    output_path = f"{DATA_DIR}/SNVs_annotated_20260715.parquet",
    add_phylop = True,
    add_next_in_frame=True,
    add_alphamissense = False,
    add_popeve = False,
    add_revel = True,
    add_clinpred = True,
    add_bayesdel = True,
    add_cpt1 = True,
    add_cadd = True,
    add_gpn_msa = True,
    add_plddt = True,
    add_pioneer = True,
    add_clinvar = True,
)

# Merge with Metadata of experimental readout type

In [ ]:
dms_meta = pl.read_csv(f"{DATA_DIR}/DMS_substitutions.csv")
dms_meta = (
    dms_meta
    .filter(pl.col('source_organism') == 'Homo sapiens')
    .rename({
        'DMS_id': 'file_name',
        'coarse_selection_type': 'exp_readout'
    })
    .select(['file_name', 'exp_readout'])
    .unique()
)
dms_meta

In [ ]:
pg_anno = pl.read_parquet(f"{DATA_DIR}/SNVs_annotated_20260715.parquet")
pg_anno

In [ ]:
pg_anno_wm = (
    dms_meta
    .join(
        pg_anno,
        on='file_name',
        how='inner',
        validate='1:m'
    )
    .rename({c: c.lower().replace(' ', '_') for c in pg_anno.columns})
    .unique()
)

# pg_anno_wm.write_parquet(f"{DATA_DIR}/SNVs_with_readout_annotated_20260716.parquet")
pg_anno_wm

In [ ]:
pg_anno_wm['exp_readout'].value_counts()